In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, classification_report
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
import joblib
from scipy import stats
from sklearn.feature_selection import RFECV
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

import os
import sys

data_path = "../../data"
os.makedirs(data_path, exist_ok=True)

plots_dir = "../../plots"
os.makedirs(plots_dir, exist_ok=True)

savedmodels_dir = "../../saved_models"
os.makedirs(savedmodels_dir, exist_ok=True)
sys.path.append(os.path.abspath(".."))


## Three Diagnosis Code-Plus-One Medication Classification

For each patient, Crohn's disease (CD; ICD-10 `K50.xxx`) and ulcerative colitis (UC; ICD-10 `K51.xxx`) diagnosis records were counted separately.

A patient was classified according to the following rules:

- **CD:** At least three CD diagnosis records, more CD records than UC records, and at least one documented exposure to a IBD medication.
- **UC:** At least three UC diagnosis records, more UC records than CD records, and at least one documented exposure to a IBD medication.
- **No IBD:** The patient did not meet the CD or UC classification criteria.

The medication criterion required at least one documented exposure to one of the prespecified nonsteroid IBD medications listed in this notebook.
'Adalimumab','Certolizumabpegol','Etrasimod','Golimumab','Infliximab','Mirikizumab','Natalizumab','Olsalazine','Ozanimod','Risankizumab','Tofacitinib','Upadacitinib','Ustekinumab','Vedolizumab','Azathioprine','Balsalazide','Mercaptopurine','Mesalamine','Methotrexate','Sulfasalazine'


In [ ]:
import pandas as pd

df = pd.read_csv(f"{data_path}/IBD_1200patients.csv")

print("df shape:", df.shape)

df = df[['PatientDurableKey','CD','UC','IBD', 'Adalimumab','Certolizumabpegol','Etrasimod','Golimumab','Infliximab','Mirikizumab','Natalizumab','Olsalazine','Ozanimod','Risankizumab','Tofacitinib','Upadacitinib','Ustekinumab','Vedolizumab','Azathioprine','Balsalazide','Mercaptopurine','Mesalamine','Methotrexate','Sulfasalazine']]

med_cols = ['Adalimumab','Certolizumabpegol','Etrasimod','Golimumab','Infliximab','Mirikizumab','Natalizumab','Olsalazine','Ozanimod','Risankizumab','Tofacitinib','Upadacitinib','Ustekinumab','Vedolizumab','Azathioprine','Balsalazide','Mercaptopurine','Mesalamine','Methotrexate','Sulfasalazine']

df["has_ibd_med"] = df[med_cols].any(axis=1).astype(int)
df

In [ ]:
import numpy as np

df["3Codes&IBDmed"] = np.select(
    [
        # CD
        (df["CD"] >= 3) &
        (df["CD"] > df["UC"]) &
        (df["has_ibd_med"] == 1),

        # UC
        (df["UC"] >= 3) &
        (df["UC"] > df["CD"]) &
        (df["has_ibd_med"] == 1),

        # No IBD
        ((df["CD"] < 3) & (df["UC"] < 3)) |
        (df["has_ibd_med"] == 0),
    ],
    [
        "CD",
        "UC",
        "No IBD"
    ],
    default="Other"
)

df

During model development, the dataset of 1,200 manually chart-reviewed patients was randomly split into a training set of 840 patients and a held-out test set of 360 patients. The same held-out test set was used to evaluate the machine learning models and the rule based three diagnosis code plus one medication definition.

In [ ]:
df1 = pd.read_csv(f"{plots_dir}/predictionresults.csv")
df1 = df1[['PatientDurableKey']]
finaldf = df1.merge(df, on='PatientDurableKey', how='left')
finaldf

In [ ]:
finaldf = finaldf[['PatientDurableKey','IBD','3Codes&IBDmed']]
threecodesIBDMed_360patientsdata = finaldf.copy()

In [ ]:
threecodesIBDMed_360patientsdata.shape

In [ ]:
y_true = threecodesIBDMed_360patientsdata['IBD']
y_pred = threecodesIBDMed_360patientsdata['3Codes&IBDmed']

In [ ]:
# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred, labels=['CD', 'No IBD','UC'])
conf_matrix_df = pd.DataFrame(conf_matrix,
                              index=['CD (Actual)', 'No IBD (Actual)','UC (Actual)'],
                              columns=['CD (Predicted)', 'No IBD (Predicted)', 'UC (Predicted)'])
print("\nConfusion Matrix:")
print(conf_matrix_df)

In [ ]:
# confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=['CD', 'No IBD', 'UC'])
print(cm)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['CD', 'No IBD', 'UC'], yticklabels=['CD', 'No IBD', 'UC'] , cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('3Codes_IBDMed')
plt.savefig(os.path.join(plots_dir, f"3codesIBDmed_confusionmatrix.pdf"),bbox_inches="tight",dpi=300)
plt.show()

In [ ]:
import numpy as np

labels = ['CD','No IBD', 'UC']
sensitivity = {}
specificity = {}
metrics = []

for i, label in enumerate(labels):
    TP = conf_matrix[i, i]  #
    FN = np.sum(conf_matrix[i, :]) - TP
    FP = np.sum(conf_matrix[:, i]) - TP
    TN = np.sum(conf_matrix) - (TP + FN + FP)

    sensitivity[label] = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity[label] = TN / (TN + FP) if (TN + FP) > 0 else 0

# Sensitivity and Specificity
print("\nSensitivity (Recall) per Class:", sensitivity)
print("Specificity per Class:", specificity)

metrics.append({
            "Sensitivity": sensitivity,
            "Specificity": specificity
        })

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(f"{plots_dir}/3codesIBDmed_sensitivityspecificty.csv", index=False)